# Credit Risk Decision Engine — Exploratory Data Analysis

This notebook preserves historical Home Credit research while explicitly auditing deployment availability. `TARGET = 1` is observed payment difficulty in the source dataset, not a universal definition of default.


In [1]:
from pathlib import Path
import sys
import warnings

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.components.data_loader import load_training_data
from src.components.data_validator import validate_training_data
from src.components.feature_contract import classify_historical_feature


## Load, validate, and audit every historical column


In [2]:
df = load_training_data()
report = validate_training_data(df)
report.raise_for_errors()
print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
availability = pd.DataFrame.from_dict(
    {column: dict(zip(["classification", "new_applicant_source"], classify_historical_feature(column))) for column in df.columns},
    orient="index",
)
display(availability.groupby("classification").size().rename("feature_count"))
display(availability)


Rows: 307,511 | Columns: 122


classification
applicant-provided               13
external-unavailable              3
historical-identifier             1
questionable-or-unavailable     100
questionable-sensitive-proxy      4
training-label                    1
Name: feature_count, dtype: int64

,classification,new_applicant_source
SK_ID_CURR,historical-identifier,Historical row traceability only; never a pred...
TARGET,training-label,Observed historical outcome; unavailable at in...
NAME_CONTRACT_TYPE,applicant-provided,Collected as Credit Product Type.
CODE_GENDER,questionable-sensitive-proxy,Excluded from the recommended deployment contr...
FLAG_OWN_CAR,applicant-provided,Collected as Owns Car.
...,...,...
AMT_REQ_CREDIT_BUREAU_DAY,questionable-or-unavailable,No reproducible source is implemented for a ne...
AMT_REQ_CREDIT_BUREAU_WEEK,questionable-or-unavailable,No reproducible source is implemented for a ne...
AMT_REQ_CREDIT_BUREAU_MON,questionable-or-unavailable,No reproducible source is implemented for a ne...
AMT_REQ_CREDIT_BUREAU_QRT,questionable-or-unavailable,No reproducible source is implemented for a ne...


`SK_ID_CURR` is a historical row identifier used only for research traceability. It is never a predictor. A deployed request receives a separate UUID-derived `application_id` after model input construction.


## Target imbalance, missingness, and employment anomaly


In [3]:
target_summary = df["TARGET"].value_counts().sort_index().to_frame("applicants")
target_summary["rate"] = df["TARGET"].value_counts(normalize=True).sort_index()
display(target_summary)
display(df.isna().mean().sort_values(ascending=False).head(30).rename("missing_rate").to_frame())
print("DAYS_EMPLOYED sentinel rows:", f"{df['DAYS_EMPLOYED'].eq(365243).sum():,}")


,applicants,rate
TARGET,,
0,282686,0.919271
1,24825,0.080729


,missing_rate
COMMONAREA_AVG,0.698723
COMMONAREA_MODE,0.698723
COMMONAREA_MEDI,0.698723
NONLIVINGAPARTMENTS_MEDI,0.694330
NONLIVINGAPARTMENTS_MODE,0.694330
NONLIVINGAPARTMENTS_AVG,0.694330
FONDKAPREMONT_MODE,0.683862
LIVINGAPARTMENTS_AVG,0.683550
LIVINGAPARTMENTS_MEDI,0.683550
LIVINGAPARTMENTS_MODE,0.683550


DAYS_EMPLOYED sentinel rows: 55,374


## Financial and categorical context


In [4]:
financial = ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE"]
display(df[financial].describe(percentiles=[0.50, 0.90, 0.99, 0.999]).T)
for column in ["NAME_CONTRACT_TYPE", "NAME_INCOME_TYPE", "NAME_HOUSING_TYPE"]:
    display(df.groupby(column, dropna=False)["TARGET"].agg(default_rate="mean", applicants="size"))


,count,mean,std,min,50%,90%,99%,99.9%,max
AMT_INCOME_TOTAL,307511.0,168797.919297,237123.146279,25650.0,147150.0,270000.0,472500.0,900000.0,117000000.0
AMT_CREDIT,307511.0,599025.999706,402490.776996,45000.0,513531.0,1133748.0,1854000.0,2517300.0,4050000.0
AMT_ANNUITY,307499.0,27108.573909,14493.737315,1615.5,24903.0,45954.0,70006.5,110047.5,258025.5
AMT_GOODS_PRICE,307233.0,538396.207429,369446.460540,40500.0,450000.0,1093500.0,1800000.0,2250000.0,4050000.0


,default_rate,applicants
NAME_CONTRACT_TYPE,,
Cash loans,0.083459,278232
Revolving loans,0.054783,29279


,default_rate,applicants
NAME_INCOME_TYPE,,
Businessman,0.000000,10
Commercial associate,0.074843,71617
Maternity leave,0.400000,5
Pensioner,0.053864,55362
State servant,0.057550,21703
Student,0.000000,18
Unemployed,0.363636,22
Working,0.095885,158774


,default_rate,applicants
NAME_HOUSING_TYPE,,
Co-op apartment,0.079323,1122
House / apartment,0.077957,272868
Municipal apartment,0.085397,11183
Office apartment,0.065724,2617
Rented apartment,0.123131,4881
With parents,0.116981,14840


## Historical external-score research


In [5]:
external = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
display(df.groupby("TARGET")[external].mean())
display(df[external].isna().mean().rename("missing_rate").to_frame())
correlations = df[[*external, "TARGET"]].corr()["TARGET"].drop("TARGET")
display(correlations.rename("correlation_with_target").to_frame())


,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3
TARGET,,,
0,0.511461,0.523479,0.520969
1,0.386968,0.410935,0.390717


,missing_rate
EXT_SOURCE_1,0.563811
EXT_SOURCE_2,0.002146
EXT_SOURCE_3,0.198253


,correlation_with_target
EXT_SOURCE_1,-0.155317
EXT_SOURCE_2,-0.160472
EXT_SOURCE_3,-0.178919


Predictive usefulness does not imply deployability. The external scores are anonymized and strong offline signals, but their generation mechanism is unavailable. They remain visible in research and are excluded from the deployable application model rather than being permanently imputed.


## Sensitive and questionable fields

Gender, family status, education, and occupation can be protected attributes or socioeconomic proxies and are excluded from the recommended production contract. Age, household composition, housing, income type, and asset ownership can also require jurisdiction-specific necessity and fairness review. This project does not claim real-world lending suitability.
